# Sweep analysis

Reads `runs.jsonl` produced by `pixi run sweep` (or `python -m experiments.sweep`).
Each row is one training run: config, seed, final losses, parameter count, and loss curves.
Swept keys are detected automatically as any config field with more than one value across runs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'axes.grid': True,
    'grid.color': '#e1e0d9',
    'axes.edgecolor': '#c3c2b7',
    'axes.labelcolor': '#52514e',
    'xtick.color': '#898781',
    'ytick.color': '#898781',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
PALETTE = ['#2a78d6', '#1baf7a', '#eda100', '#008300', '#4a3aa7', '#e34948', '#e87ba4', '#eb6834']

RUNS = 'runs.jsonl'
raw = pd.read_json(RUNS, lines=True)
cfg = pd.json_normalize(raw['config']).add_prefix('cfg.')
df = pd.concat([raw.drop(columns='config'), cfg], axis=1)
swept = [c for c in cfg.columns if df[c].nunique() > 1]
assert swept, 'only one config in runs.jsonl - nothing to compare yet'
print(f'{len(df)} runs; swept keys: {swept}')
df.tail(3)

In [ ]:
summary = (df.groupby(swept)
             .agg(val_loss=('val_loss', 'mean'),
                  val_std=('val_loss', 'std'),
                  num_params=('num_params', 'first'),
                  seeds=('seed', 'count'))
             .sort_values('val_loss'))
summary.round(4)

In [ ]:
if {'cfg.dim_embed', 'cfg.dim_hidden'} <= set(swept):
    display(df.pivot_table(index='cfg.dim_embed', columns='cfg.dim_hidden',
                           values='val_loss', aggfunc='mean').round(4))

In [ ]:
per_cfg = summary.reset_index()
labels = per_cfg[swept].astype(str).agg(', '.join, axis=1)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(per_cfg['num_params'], per_cfg['val_loss'], color=PALETTE[0], s=48)
for xy, label in zip(per_cfg[['num_params', 'val_loss']].values, labels):
    ax.annotate(label, xy, textcoords='offset points', xytext=(6, 4),
                fontsize=8, color='#52514e')
ax.set_xlabel('parameters')
ax.set_ylabel('val loss (mean over seeds)')
ax.set_title(f"Val loss vs model size ({', '.join(k.removeprefix('cfg.') for k in swept)})")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for color, key in zip(PALETTE[:4], summary.index):
    key = key if isinstance(key, tuple) else (key,)
    mask = np.logical_and.reduce([(df[c] == v).values for c, v in zip(swept, key)])
    runs = df[mask]
    iters = runs.iloc[0]['curves']['iter']
    val = np.mean([c['val_loss'] for c in runs['curves']], axis=0)
    label = ', '.join(f"{c.removeprefix('cfg.')}={v}" for c, v in zip(swept, key))
    ax.plot(iters, val, color=color, linewidth=2, label=label)
ax.set_xlabel('iteration')
ax.set_ylabel('val loss (mean over seeds)')
ax.set_title('Val loss curves, best 4 configs')
ax.legend(frameon=False)
plt.tight_layout()